<span style="color:pink; font-size:15px;">File to find the optimal unitaries for the LTO protocol using optimization techniques for gaining the maximum ergotropy advantage</span>

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from state_gen import generate_state
# from energy import energy, free_energy, passive_energy_g, passive_energy_l, ergotropy_gap, global_ergo, local_ergo
from local_to import qubit_ham, sho_ham, gibbs_state, LTO_step
# from unitary import deg_unitary
from optimize import search_over_states, random_search, nelder_mead_search, lbfgs_search
from tqdm import tqdm
np.set_printoptions(linewidth=np.inf, precision=4, suppress=True)

def fmt(M): return np.array2string(M, precision=4, suppress_small=True, floatmode='fixed', max_line_width=80)

In [ ]:
# Default Parameters

# Inverse temperatures and dimensions
beta_a=1.0
beta_b=0.9
bath_dim = 8

# Hamiltonian parameters
w1 = 0.003
w2 = 0.001
omega_a = omega_b = 0.01

# State generation parameters
a = 0.8
p = 0.5
n_terms = 4
N = np.sqrt(a * (1 - a))

kind_dict = {
    1: "product",
    2: "separable",
    3: "schmidt_ent",
    4: "pure_ent",
    5: "werner",
    6: "mixed_ent",
    7: "random"
}

In [ ]:
# n_states=50; n_unitaries=200
# search_over_states(n_states, n_unitaries, beta_a, beta_b, w1, w2, omega_a, omega_b, bath_dim)

In [ ]:
# Precompute Hamiltonians and Gibbs states for baths and systems
Hs1 = qubit_ham(w1);          Hs2 = qubit_ham(w2)
Hb1 = sho_ham(bath_dim, omega_a); Hb2 = sho_ham(bath_dim, omega_b)
gamma_Ra = gibbs_state(Hb1, beta_a)
gamma_Rb = gibbs_state(Hb2, beta_b)

# master tracker across all states
master_best = {
    'global': (-np.inf, None, None, None, None),  # (delta, rho12, sigma12, Ua, Ub)
    'local':  (-np.inf, None, None, None, None),
    'gap':    (-np.inf, None, None, None, None),
}

n_states    = 20
n_restarts  = 10
maxiter     = 500

for s in range(n_states):
    print(f"\n{'='*50}")
    print(f"  STATE {s+1}/{n_states}")
    print(f"{'='*50}")

    rho12 = generate_state(kind='random')

    # run both optimizers
    best_nm   = nelder_mead_search(rho12, gamma_Ra, gamma_Rb, Hs1, Hs2, Hb1, Hb2, w1, w2, n_restarts, maxiter)
    best_lbfgs = lbfgs_search(rho12, gamma_Ra, gamma_Rb, Hs1, Hs2, Hb1, Hb2, w1, w2, n_restarts, maxiter)

    # update master tracker
    for target in ['global', 'local', 'gap']:
        for best, label in [(best_nm, 'NM'), (best_lbfgs, 'LBFGS')]:
            delta, sigma12, Ua, Ub = best[target]
            if delta > master_best[target][0]:
                master_best[target] = (delta, rho12.copy(),
                                       sigma12.copy(), Ua.copy(), Ub.copy())
                print(f"  ✓ New best Δ({target}) = {delta:.6f} [{label}]")

# final summary
print(f"\n{'='*50}")
print("  FINAL BEST ACROSS ALL STATES")
print(f"{'='*50}")
for target in ['global', 'local', 'gap']:
    delta = master_best[target][0]
    print(f"  Δ({target:6s}) = {delta:.6f}  "
          f"{'✓ > 0' if delta > 1e-8 else '✗ not found'}")
print(f"{'='*50}")


  STATE 1/20


Nelder-Mead:  10%|█         | 1/10 [00:23<03:29, 23.27s/restart]

  restart   1  Δglobal=-0.000001  Δlocal=-0.000001  Δgap=0.000000


Nelder-Mead:  20%|██        | 2/10 [00:46<03:06, 23.28s/restart]

  restart   2  Δglobal=-0.000001  Δlocal=-0.000001  Δgap=0.000000


...

  restart   8  Δglobal=-0.000001  Δlocal=-0.000002  Δgap=-0.000001


L-BFGS-B:  90%|█████████ | 9/10 [01:55<00:13, 13.33s/restart]

  restart   9  Δglobal=-0.000001  Δlocal=-0.000002  Δgap=-0.000001


L-BFGS-B: 100%|██████████| 10/10 [01:58<00:00, 11.90s/restart]

  restart  10  Δglobal=-0.000001  Δlocal=-0.000002  Δgap=-0.000001

  L-BFGS-B RESULTS
  Best Δ(global) = -0.000001  ✗ not found
  Best Δ(local ) = -0.000002  ✗ not found
  Best Δ(gap   ) = -0.000001  ✗ not found

  FINAL BEST ACROSS ALL STATES
  Δ(global) = -0.000000  ✗ not found
  Δ(local ) = 0.000000  ✗ not found
  Δ(gap   ) = 0.001049  ✓ > 0
